# 🚀 PyTorch Deep Learning Workshop: Jet Classification

Welcome to the PyTorch Classification Workshop! This hands-on notebook is designed to guide you through the entire PyTorch pipeline in about 30 to 45 minutes.

Rather than just running finished code, you'll be actively modifying the architecture, tuning hyperparameters, and diagnosing training performance—just like a real-world machine learning engineer.

### 🎯 The Task: LHC Jet Classification
We will be working with a real-world scientific dataset: **hls4ml Jet High-Level Features (`hls4ml_HLF.arff`)**. 
Our goal is to classify subatomic particle jets produced in high-energy collisions at the Large Hadron Collider (LHC) into one of **5 categories**:
- `g`: Gluons (0)
- `q`: Quarks (1)
- `w`: W bosons (2)
- `z`: Z bosons (3)
- `t`: Top quarks (4)

We have **16 High-Level Features** representing shape, mass, energy distribution, and jet multiplicity.

---

## Imports & Hardware Check
First, let's import the necessary packages and check our hardware acceleration (CUDA for Nvidia GPUs, MPS for Apple Silicon, or CPU).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA (NVIDIA GPU) available: {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon GPU) available: {torch.backends.mps.is_available()}")

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

### 🧪Exercise 1 
1. **What is `nn` vs `F`?** Why does PyTorch separate them? (Hint: Think about stateful object-oriented layers vs stateless mathematical functions).
2. **Why do we use `DataLoader`** instead of just feeding the entire dataset into the model at once? What memory and optimization benefits does it provide?
3. **What is CUDA or MPS?** Why is GPU training so much faster than CPU training for deep learning?

## Load and Prepare the Dataset
We load the ARFF file, decode byte labels into strings, map them to numeric values, take a random subset for speed, split them, normalize them, and finally convert them to PyTorch tensors.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

print("Loading hls4ml_HLF.arff (this might take ~5-10 seconds)...")
data, meta = arff.loadarff('hls4ml_HLF.arff')
df = pd.DataFrame(data)

# Decode bytes to strings for labels
df['class'] = df['class'].str.decode('utf-8')

# Class mapping
class_names = ['g', 'q', 'w', 'z', 't']
class_mapping = {name: idx for idx, name in enumerate(class_names)}
df['label'] = df['class'].map(class_mapping)

# Extract features and labels
feature_cols = [col for col in df.columns if col not in ['class', 'label']]
X = df[feature_cols].values
y = df['label'].values

# --- WORKSHOP SUBSAMPLING ---
# The full dataset has 830,000 samples. Training on all of them on a CPU could take 20+ minutes.
# We will use a fast subset of 20,000 samples for the workshop. 
# You can increase this to 100,000+ or the full dataset later to boost your final accuracy!
n_samples = 20000
indices = np.random.choice(len(X), n_samples, replace=False)
X_subset = X[indices]
y_subset = y[indices]

# Split into Train (80%) and Test (20%)
train_X_raw, test_X_raw, train_y, test_y = train_test_split(
    X_subset, y_subset, test_size=0.2, random_state=42, stratify=y_subset
)

# --- FEATURE NORMALIZATION ---
# Neural networks perform best when input features are standardized (zero mean, unit variance).
scaler = StandardScaler()
train_X_scaled = scaler.fit_transform(train_X_raw)
test_X_scaled = scaler.transform(test_X_raw)

# Convert to PyTorch tensors
train_X = torch.tensor(train_X_scaled, dtype=torch.float32)
test_X = torch.tensor(test_X_scaled, dtype=torch.float32)
train_Y = torch.tensor(train_y, dtype=torch.long)
test_Y = torch.tensor(test_y, dtype=torch.long)

print("\n--- Dataset Summary ---")
print(f"Total features: {train_X.shape[1]}")
print(f"Training samples: {train_X.shape[0]}")
print(f"Test samples: {test_X.shape[0]}")
print(f"Class counts in subset:\n{pd.Series(y_subset).map({v:k for k,v in class_mapping.items()}).value_counts()}")

### 🧪 Exercise 2 — Class Overlap & Normalization:
1. **What happens if classes overlap significantly?** How does that affect the theoretical maximum accuracy our model can achieve?
2. **Why do we normalize features using `StandardScaler`?** What would happen to the weights in the first layer if one feature ranged from `0` to `1` and another ranged from `0` to `1,000,000`? (Hint: Think about gradients and learning rates).
3. **Examine the class distributions.** Are they balanced or unbalanced? How does class imbalance affect evaluation metrics (e.g., standard accuracy vs balanced accuracy)?

## Define the Model Architecture
Here we define our custom neural network class by inheriting from `nn.Module`. 
In PyTorch, we define our layers in `__init__` and the network's forward logic in `forward`.

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        # Layer 1: Linear projection from inputs to hidden features
        self.layer1 = nn.Linear(input_dim, hidden_dim)
        # Non-linear activation
        self.relu = nn.ReLU()
        # Layer 2: Projection from hidden features to class logits
        self.layer2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.layer2(x)
        return x

# Instantiate model
model = SimpleClassifier(
    input_dim=16,       # 16 High-Level Features
    hidden_dim=32,      # Size of the hidden layer representation
    num_classes=5       # 5 jet classes
)

print(model)
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

### 🛠️ Exercise 3 — Network Architecture & Parameters
1. **Parameter Scaling**: Change `hidden_dim` from `32` to `8`, `64`, `128`, and `256`. Note down how the parameter count changes. Can you compute the formula for the number of parameters in this model? (Hint: don't forget biases!)
2. **Going Deeper (Coding Challenge)**: Modify the `SimpleClassifier` class (or write a new class `DeepClassifier` below) to add a second hidden layer. Your network flow should be:
   * `Linear(input_dim -> hidden_dim)`
   * `ReLU()`
   * `Linear(hidden_dim -> hidden_dim)`
   * `ReLU()`
   * `Linear(hidden_dim -> num_classes)`
3. **Dropout Regularization (Coding Challenge)**: Insert a dropout layer (`nn.Dropout(p=0.3)`) after the activation function(s) to mitigate overfitting. What does dropout do during training, and how does it behave during evaluation (`model.eval()`)?

## DataLoader, Optimizer, and Loss
Next, we prepare our `DataLoader` for training, and choose our loss function and optimizer. 
These three components dictate how batches are loaded, how errors are quantified, and how weights are adjusted.

In [ ]:
# Create DataLoader to feed data in mini-batches during training
train_loader = DataLoader(
    TensorDataset(train_X, train_Y),
    batch_size=32,      # Feed 32 samples at a time
    shuffle=True        # Shuffle every epoch to prevent ordering bias
)

# Loss function: Multi-class Cross Entropy
criterion = nn.CrossEntropyLoss()

# Optimizer: Adam optimizer with learning rate 0.01
learning_rate = 0.01
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

### Exercise 4 — Hyperparameter Impact
1. **Batch Size Sweep**: What happens if you change `batch_size` to `8`, `16`, `64`, or `128`? How does batch size affect training speed (seconds per epoch) and the smoothness of the loss curve?
2. **Optimizer Experimentation**: Replace `Adam` with standard stochastic gradient descent (`torch.optim.SGD(model.parameters(), lr=0.01)`). Does standard SGD learn faster or slower than Adam? Why does Adam converge faster in complex settings?
3. **Learning Rate Search**: Try changing `lr` to:
   * `0.1`: Does the loss explode or oscillate wildly?
   * `0.0001`: Does the model learn too slowly?
4. **L2 Regularization**: Add `weight_decay=1e-4` to the `Adam` optimizer. Try values of `0`, `1e-5`, `1e-3`, and `1e-2`. How does L2 weight decay combat overfitting?

## The Training Loop
The heart of deep learning in PyTorch is the training loop. We iterate over our epochs and mini-batches, performing forward passes, gradient calculations, and optimizer steps.

In [ ]:
epochs = 20

# Move model to the selected hardware device (GPU or CPU)
model = model.to(device)

train_losses = []
train_accs = []

for epoch in range(epochs):
    model.train() # Set model to training mode (enables Dropout/BatchNorm)
    
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_Y in train_loader:
        # Move mini-batch to active device
        batch_X, batch_Y = batch_X.to(device), batch_Y.to(device)
        
        # 1. Zero out previous gradients
        optimizer.zero_grad()
        
        # 2. Forward pass: compute predictions (logits)
        logits = model(batch_X)
        
        # 3. Compute loss
        loss = criterion(logits, batch_Y)
        
        # 4. Backward pass: compute gradients of loss w.r.t parameters
        loss.backward()
        
        # 5. Optimizer step: update weights
        optimizer.step()
        
        # Collect batch statistics
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == batch_Y).sum().item()
        total += len(batch_Y)
        
    # Calculate epoch metrics
    epoch_loss = total_loss / len(train_loader)
    epoch_acc = 100.0 * correct / total
    
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    
    print(f"Epoch {epoch+1:02d}/{epochs:02d} | Loss: {epoch_loss:.4f} | Train Accuracy: {epoch_acc:.2f}%")

### Exercise 5 — Training Loop Dynamics & Early Stopping
1. **Underfitting vs. Overfitting**: Run the training for `5`, `20`, and `100` epochs. At what point does the training loss stop decreasing significantly? Does the model begin to overfit if trained for too long?
2. **Early Stopping Coding Challenge**: Modify the training loop below (or implement a new version) to incorporate **Early Stopping**. 
   * *Instruction:* Evaluate validation loss on the test set at the end of each epoch. Keep track of the best validation loss. If the validation loss does not improve for 5 consecutive epochs, print a message and break the loop early.

## Evaluate the Model
We must test our model on unseen data. During evaluation, we put the model in `.eval()` mode and wrap our code in `with torch.no_grad():` to turn off gradient computation (saving memory and compute).

In [ ]:
model.eval() # Set model to evaluation mode (disables Dropout/BatchNorm)

# Move test dataset to the active device
test_X = test_X.to(device)
test_Y = test_Y.to(device)

# Disable gradient computations
with torch.no_grad():
    logits = model(test_X)
    preds = logits.argmax(dim=1)
    acc = (preds == test_Y).float().mean()

print(f"Test Accuracy: {acc.item() * 100:.2f}%")

### Error Analysis & Confusion Matrix
1. **Inspect Predictions (Coding challenge)**: Write a quick snippet to print the first 10 predictions alongside their true labels. Identify which predictions are correct and which are wrong.
2. **Locate Misclassifications (Coding challenge)**: Print the index and feature values of 3 samples that the model predicted incorrectly. What might have confused the model?
3. **Confusion Matrix (Coding challenge)**: Use `sklearn.metrics.confusion_matrix` and `matplotlib.pyplot` to compute and plot a Confusion Matrix. 
   * *Questions:* Which particle classes are most frequently confused with each other? (e.g., quark `q` vs gluon `g`, or W boson `w` vs Z boson `z`?) Why does this make physical sense?

## Logits vs Softmax Probabilities
Our model outputs raw values called **logits**. To convert them into interpretable probabilities that sum to 1, we apply the Softmax activation.

In [ ]:
# Apply Softmax along the class dimension (dim=1)
probabilities = torch.softmax(logits, dim=1)

# Display the first 5 test samples
for i in range(5):
    print(f"Sample {i+1}:")
    print(f"  Logits:        {logits[i].cpu().numpy()}")
    print(f"  Probabilities: {probabilities[i].cpu().numpy()} (Sum: {probabilities[i].sum().item():.2f})")
    print(f"  Prediction:    {class_names[preds[i].item()]} (Class {preds[i].item()})")
    print(f"  True Label:    {class_names[test_Y[i].item()]} (Class {test_Y[i].item()})\n")

### 🤯 Exercise 7 — The Softmax Mystery
1. **CrossEntropyLoss Detail**: Look back at Cell 4 where we defined our loss function as `nn.CrossEntropyLoss()`. Notice that our model's last layer in Cell 3 is a `nn.Linear` layer that directly outputs raw logits (not probabilities).
2. **Why does `nn.CrossEntropyLoss` NOT want us to add a Softmax layer at the end of our model?** 
   * *Hint:* Read the PyTorch documentation. What two operations are combined inside `nn.CrossEntropyLoss`? (Answer: LogSoftmax and Negative Log Likelihood Loss - NLLLoss).
3. **What is the numerical stability benefit** of combining Softmax and Log operations together instead of computing them separately? (Hint: Think about floating-point exponentiation overflow and underflow).

## Save and Load the Model
Once you have trained your model, you'll want to save its weights so you can deploy it later. In PyTorch, we save the `state_dict()` which contains the model's weight and bias matrices.

In [ ]:
# Save the trained weights to a file
torch.save(model.state_dict(), "classifier.pt")
print("Model weights saved to classifier.pt!")

# To load, we must first instantiate the architecture
loaded_model = SimpleClassifier(input_dim=16, hidden_dim=32, num_classes=5)

# Load the weights into the architecture
loaded_model.load_state_dict(torch.load("classifier.pt"))
loaded_model = loaded_model.to(device)
loaded_model.eval()

# Double check validation accuracy matches exactly
with torch.no_grad():
    loaded_logits = loaded_model(test_X)
    loaded_preds = loaded_logits.argmax(dim=1)
    loaded_acc = (loaded_preds == test_Y).float().mean()

print(f"Loaded Model Test Accuracy: {loaded_acc.item() * 100:.2f}%")

### 🏆 Exercise 8 — The Ultimate LHC Jet Classification Challenge!
Now it's time to put everything you've learned to the test! 
Your objective is to modify the code across the cells (or write custom code in the cell below) to achieve the **highest possible test accuracy** on the jet classification dataset.

#### 🛠️ Hyperparameter Tuning Strategies to Explore:
1. **Scale up the Data**: Increase `n_samples` from `20000` to `50000`, `150000`, or use the entire `830,000` dataset (in Cell 2).
2. **Wider/Deeper Model**: Modify the architecture (in Cell 3) to add more layers and hidden units. (e.g. layers of size `128` and `64`).
3. **Combat Overfitting**: Introduce `nn.Dropout(p=0.2)` or `nn.Dropout(p=0.4)` and L2 regularization (`weight_decay=1e-4` in the optimizer in Cell 4).
4. **Learning Rate Scheduling**: Keep your learning rate high initially, and decay it as training progresses.
5. **Batch Size Tuning**: Try different batch sizes (`16`, `32`, `64`, `128`, `256`).
6. **Optimizer Experimentation**: Try `Adam`, `AdamW`, `RMSprop`, or `SGD` with momentum.

Use the scratch cell below to design, train, and validate your optimized model. Can you surpass **80% accuracy**? Share your architecture, training strategy, and best test accuracy with the class!

In [ ]:
# Write and run your custom challenge code here!
# Hint: Define a new model, configure a custom DataLoader & optimizer, train and evaluate.

